# Threat Intelligence Aggregator
**IBM SkillsBuild Cyber Security 2026 – conducted by BharatCares**

A Python tool that pulls live Indicators of Compromise (IoCs) from **AlienVault OTX** and enriches them with
reputation data from **VirusTotal**, then merges both sources into one normalised, scored report.

**Pipeline:** `Collect IoCs → Detect type → Query VirusTotal + OTX → Normalise → Risk score → Report / Export`

> **Ethical use:** This tool only performs *passive, read-only* look-ups against public threat-intel APIs.
> Never scan or attack systems you do not own. Respect each API's terms of service and rate limits.

## 1. Setup and imports

In [ ]:
import os, re, time, json, base64, ipaddress, logging, random, hashlib
from datetime import datetime, timedelta, timezone

import requests
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Load API keys from a local .env file if python-dotenv is installed (optional)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("threat-intel")

## 2. Configuration
API keys are read from **environment variables** (never hard-code them in the notebook):

| Variable | Where to get it |
|---|---|
| `VT_API_KEY` | https://www.virustotal.com → profile → *API key* (free tier: 4 requests/min, 500/day) |
| `OTX_API_KEY` | https://otx.alienvault.com → *Settings* → *OTX Key* (free) |

If **no keys are found**, the notebook automatically switches to **DEMO MODE** with simulated responses so
the whole pipeline can still be demonstrated offline.

In [ ]:
VT_API_KEY  = os.getenv("VT_API_KEY", "").strip()
OTX_API_KEY = os.getenv("OTX_API_KEY", "").strip()

VT_BASE  = "https://www.virustotal.com/api/v3"
OTX_BASE = "https://otx.alienvault.com/api/v1"

VT_CALLS_PER_MIN = 4          # VirusTotal free-tier limit
MAX_IOCS         = 12         # how many IoCs to analyse per run
FEED_HOURS       = 24         # pull OTX pulses modified in the last N hours
OUTPUT_DIR       = "output"

DEMO_MODE = not (VT_API_KEY or OTX_API_KEY)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("VirusTotal key :", "found" if VT_API_KEY else "missing")
print("AlienVault key :", "found" if OTX_API_KEY else "missing")
print("Mode           :", "DEMO (simulated data)" if DEMO_MODE else "LIVE")

## 3. IoC helpers – type detection, validation and defanging

In [ ]:
RE_MD5    = re.compile(r"^[a-fA-F0-9]{32}$")
RE_SHA1   = re.compile(r"^[a-fA-F0-9]{40}$")
RE_SHA256 = re.compile(r"^[a-fA-F0-9]{64}$")
RE_DOMAIN = re.compile(r"^(?=.{1,253}$)(?!-)([a-z0-9-]{1,63}(?<!-)\.)+[a-z]{2,63}$", re.I)
RE_URL    = re.compile(r"^https?://\S+$", re.I)


def refang(value: str) -> str:
    """Convert a defanged IoC (hxxp://evil[.]com) back to its normal form."""
    return (value.strip().replace("[.]", ".").replace("(.)", ".")
            .replace("hxxp", "http").replace("[:]", ":"))


def defang(value: str) -> str:
    """Make an IoC safe to print/share (evil.com -> evil[.]com)."""
    return value.replace("http", "hxxp").replace(".", "[.]")


def detect_ioc_type(value: str):
    """Return 'ip', 'domain', 'url', 'hash' or None if the value is not a valid IoC."""
    v = refang(value)
    try:
        ipaddress.ip_address(v)
        return "ip"
    except ValueError:
        pass
    if RE_URL.match(v):
        return "url"
    if RE_MD5.match(v) or RE_SHA1.match(v) or RE_SHA256.match(v):
        return "hash"
    if RE_DOMAIN.match(v):
        return "domain"
    return None


# quick self-test
for sample in ["8.8.8.8", "hxxp://example[.]com/a", "example.com",
               "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f", "not an ioc"]:
    print(f"{sample[:45]:<48} -> {detect_ioc_type(sample)}")

## 4. API clients
A small rate limiter and a resilient `http_get` (timeouts, retries with back-off, HTTP 429 handling) are shared by both clients.

In [ ]:
class RateLimiter:
    """Blocks so that no more than `calls_per_minute` calls are made."""
    def __init__(self, calls_per_minute: int):
        self.interval = 60.0 / calls_per_minute
        self._last = 0.0

    def wait(self):
        delay = self._last + self.interval - time.monotonic()
        if delay > 0:
            time.sleep(delay)
        self._last = time.monotonic()


def http_get(url, headers, params=None, retries=3, timeout=20):
    """GET with retries. Returns parsed JSON, or None for 404 / repeated failure."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None                              # IoC not known to this service
            if r.status_code == 429:                     # rate limited -> back off
                wait = 15 * (attempt + 1)
                log.warning("Rate limited by %s, sleeping %ss", url.split('/')[2], wait)
                time.sleep(wait)
                continue
            if r.status_code in (401, 403):
                log.error("Authentication failed for %s - check your API key.", url.split('/')[2])
                return None
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("Request error (%s) attempt %d/%d", exc, attempt + 1, retries)
            time.sleep(2 ** attempt)
    return None

In [ ]:
class VirusTotalClient:
    """Reputation look-ups via the VirusTotal v3 API."""
    def __init__(self, api_key):
        self.headers = {"x-apikey": api_key, "accept": "application/json"}
        self.limiter = RateLimiter(VT_CALLS_PER_MIN)

    def _endpoint(self, ioc, ioc_type):
        if ioc_type == "ip":     return f"{VT_BASE}/ip_addresses/{ioc}"
        if ioc_type == "domain": return f"{VT_BASE}/domains/{ioc}"
        if ioc_type == "hash":   return f"{VT_BASE}/files/{ioc}"
        if ioc_type == "url":
            url_id = base64.urlsafe_b64encode(ioc.encode()).decode().strip("=")
            return f"{VT_BASE}/urls/{url_id}"
        raise ValueError(ioc_type)

    def lookup(self, ioc, ioc_type):
        self.limiter.wait()
        data = http_get(self._endpoint(ioc, ioc_type), self.headers)
        if not data:
            return None
        attrs = data.get("data", {}).get("attributes", {})
        stats = attrs.get("last_analysis_stats", {})
        return {
            "vt_malicious":  stats.get("malicious", 0),
            "vt_suspicious": stats.get("suspicious", 0),
            "vt_harmless":   stats.get("harmless", 0),
            "vt_undetected": stats.get("undetected", 0),
            "vt_reputation": attrs.get("reputation"),
            "country":       attrs.get("country"),
            "as_owner":      attrs.get("as_owner"),
        }


class OTXClient:
    """Pulse feed and indicator look-ups via AlienVault OTX (DirectConnect API)."""
    TYPE_TO_OTX = {"ip": "IPv4", "domain": "domain", "hash": "file", "url": "url"}
    # OTX indicator type -> our simplified type
    OTX_TO_TYPE = {"IPv4": "ip", "IPv6": "ip", "domain": "domain", "hostname": "domain", "URL": "url",
                   "FileHash-MD5": "hash", "FileHash-SHA1": "hash", "FileHash-SHA256": "hash"}

    def __init__(self, api_key):
        self.headers = {"X-OTX-API-KEY": api_key, "accept": "application/json"}

    def latest_iocs(self, hours=24, limit=50):
        """Live feed: IoCs from pulses you subscribe to that changed in the last `hours` hours."""
        since = (datetime.now(timezone.utc) - timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%S")
        data = http_get(f"{OTX_BASE}/pulses/subscribed", self.headers,
                        params={"limit": 20, "modified_since": since}) or {}
        found, seen = [], set()
        for pulse in data.get("results", []):
            for ind in pulse.get("indicators", []):
                ioc_type = self.OTX_TO_TYPE.get(ind.get("type"))
                value = ind.get("indicator")
                if ioc_type and value and value not in seen:
                    seen.add(value)
                    found.append({"ioc": value, "type": ioc_type, "feed_pulse": pulse.get("name", "")[:60]})
                if len(found) >= limit:
                    return found
        return found

    def lookup(self, ioc, ioc_type):
        otx_type = self.TYPE_TO_OTX[ioc_type]
        if ioc_type == "ip" and ":" in ioc:
            otx_type = "IPv6"
        data = http_get(f"{OTX_BASE}/indicators/{otx_type}/{ioc}/general", self.headers)
        if not data:
            return None
        pulse_info = data.get("pulse_info", {})
        tags = {t for p in pulse_info.get("pulses", []) for t in p.get("tags", [])}
        families = {mf.get("display_name") for p in pulse_info.get("pulses", [])
                    for mf in p.get("malware_families", []) if isinstance(mf, dict)}
        return {
            "otx_pulses":  pulse_info.get("count", 0),
            "otx_tags":    ", ".join(sorted(tags)[:5]),
            "otx_malware": ", ".join(sorted(f for f in families if f)[:3]),
            "country":     data.get("country_name"),
        }

## 5. Demo-mode data source
Used automatically when no API keys are set. Responses are **simulated** (deterministic per IoC) so the
notebook runs offline – results from this mode are *not* real threat intelligence.

In [ ]:
DEMO_IOCS = [
    {"ioc": "8.8.8.8", "type": "ip", "feed_pulse": "demo: known-good resolver"},
    {"ioc": "example.com", "type": "domain", "feed_pulse": "demo: known-good domain"},
    {"ioc": "203.0.113.45", "type": "ip", "feed_pulse": "demo: botnet C2 (TEST-NET-3)"},
    {"ioc": "198.51.100.77", "type": "ip", "feed_pulse": "demo: scanning activity (TEST-NET-2)"},
    {"ioc": "login-secure-update[.]example", "type": "domain", "feed_pulse": "demo: phishing campaign"},
    {"ioc": "http://payments-verify.example/login.php", "type": "url", "feed_pulse": "demo: phishing campaign"},
    {"ioc": "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f", "type": "hash",
     "feed_pulse": "demo: EICAR antivirus test file"},
    {"ioc": "44d88612fea8a8f36de82e1278abb02f", "type": "hash", "feed_pulse": "demo: EICAR (MD5)"},
    {"ioc": "192.0.2.10", "type": "ip", "feed_pulse": "demo: brute-force source (TEST-NET-1)"},
    {"ioc": "update-flashplayer[.]example", "type": "domain", "feed_pulse": "demo: malware dropper"},
    {"ioc": "cdn.example.org", "type": "domain", "feed_pulse": "demo: benign CDN"},
    {"ioc": "http://files-share.example/invoice.exe", "type": "url", "feed_pulse": "demo: malware delivery"},
]
KNOWN_GOOD = {"8.8.8.8", "example.com", "cdn.example.org"}


def _rng(ioc):
    return random.Random(int(hashlib.sha256(ioc.encode()).hexdigest(), 16))


def demo_vt(ioc, ioc_type):
    r = _rng("vt" + ioc)
    clean = ioc in KNOWN_GOOD
    mal = 0 if clean else r.randint(1, 40)
    sus = 0 if clean else r.randint(0, 6)
    return {"vt_malicious": mal, "vt_suspicious": sus, "vt_harmless": r.randint(55, 70),
            "vt_undetected": r.randint(8, 20), "vt_reputation": -mal if not clean else r.randint(50, 500),
            "country": r.choice(["US", "RU", "CN", "NL", "DE", "BR"]) if ioc_type == "ip" else None,
            "as_owner": None}


def demo_otx(ioc, ioc_type):
    r = _rng("otx" + ioc)
    clean = ioc in KNOWN_GOOD
    return {"otx_pulses": 0 if clean else r.randint(0, 15),
            "otx_tags": "" if clean else ", ".join(r.sample(["phishing", "c2", "botnet", "trojan", "scanner",
                                                              "ransomware", "dropper"], 3)),
            "otx_malware": "" if clean else r.choice(["", "AgentTesla", "Emotet", "Mirai", "RedLine"]),
            "country": None}

## 6. Aggregation and risk scoring
Each source produces a 0–100 component score which are merged into a single **risk score**.

* **VirusTotal score** = `min(100, (malicious + 0.5 × suspicious) × 5)` → 20 detections ≈ 100
* **OTX score** = `min(100, pulses × 10)` → appearing in 10 threat reports ≈ 100
* **Combined** = `0.7 × VT + 0.3 × OTX` (if only one source answered, that source's score is used)

| Score | Verdict |
|---|---|
| ≥ 70 | **High** |
| 30 – 69 | **Medium** |
| 1 – 29 | **Low** |
| 0 | **Clean / Unknown** |

In [ ]:
def vt_score(r):
    return min(100, (r.get("vt_malicious", 0) + 0.5 * r.get("vt_suspicious", 0)) * 5)

def otx_score(r):
    return min(100, r.get("otx_pulses", 0) * 10)

def verdict_for(score):
    if score >= 70: return "High"
    if score >= 30: return "Medium"
    if score >= 1:  return "Low"
    return "Clean/Unknown"


def aggregate(ioc, ioc_type, vt=None, otx=None, feed_pulse=""):
    """Merge VT + OTX results for a single IoC into one flat record."""
    row = {"ioc": defang(ioc), "type": ioc_type, "feed_pulse": feed_pulse}
    sources, scores = [], []
    if vt:
        row.update(vt); sources.append("VirusTotal"); scores.append((0.7, vt_score(vt)))
    if otx:
        row.update({k: v for k, v in otx.items() if k != "country" or not row.get("country")})
        sources.append("AlienVault OTX"); scores.append((0.3, otx_score(otx)))
    if scores:
        total_w = sum(w for w, _ in scores)
        risk = round(sum(w * s for w, s in scores) / total_w, 1)
    else:
        risk = 0.0
    row.update({"risk_score": risk, "verdict": verdict_for(risk) if scores else "No data",
                "sources": " + ".join(sources) or "none",
                "checked_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")})
    return row

## 7. Collect IoCs (live feed from OTX, or demo data)

In [ ]:
vt_client  = VirusTotalClient(VT_API_KEY) if VT_API_KEY else None
otx_client = OTXClient(OTX_API_KEY) if OTX_API_KEY else None

# Optionally add your own IoCs (IPs, domains, URLs, hashes). Defanged forms are accepted.
MANUAL_IOCS = []   # e.g. ["1.2.3.4", "evil[.]example.com"]

if DEMO_MODE:
    feed = list(DEMO_IOCS)
elif otx_client:
    feed = otx_client.latest_iocs(hours=FEED_HOURS, limit=MAX_IOCS)
    log.info("OTX returned %d IoCs from the last %dh", len(feed), FEED_HOURS)
else:
    feed = []   # only a VirusTotal key -> analyse MANUAL_IOCS only

for m in MANUAL_IOCS:
    t = detect_ioc_type(m)
    if t:
        feed.append({"ioc": refang(m), "type": t, "feed_pulse": "manual"})
    else:
        log.warning("Skipping invalid IoC: %s", m)

feed = feed[:MAX_IOCS]
print(f"{len(feed)} IoCs queued for enrichment")
pd.DataFrame(feed)

## 8. Enrich and aggregate

In [ ]:
records = []
for i, item in enumerate(feed, 1):
    ioc = refang(item["ioc"])
    ioc_type = item.get("type") or detect_ioc_type(ioc)
    log.info("[%d/%d] %s (%s)", i, len(feed), defang(ioc), ioc_type)

    if DEMO_MODE:
        vt, otx = demo_vt(ioc, ioc_type), demo_otx(ioc, ioc_type)
    else:
        vt  = vt_client.lookup(ioc, ioc_type)  if vt_client  else None
        otx = otx_client.lookup(ioc, ioc_type) if otx_client else None

    records.append(aggregate(ioc, ioc_type, vt, otx, item.get("feed_pulse", "")))

df = (pd.DataFrame(records)
        .sort_values("risk_score", ascending=False)
        .reset_index(drop=True))
cols = ["ioc", "type", "verdict", "risk_score", "vt_malicious", "vt_suspicious",
        "otx_pulses", "otx_tags", "otx_malware", "country", "sources", "feed_pulse"]
df[[c for c in cols if c in df.columns]]

## 9. Summary and visualisation

In [ ]:
print("Mode :", "DEMO (simulated data)" if DEMO_MODE else "LIVE")
print("\nIoCs by verdict:")
print(df["verdict"].value_counts().to_string())
print("\nIoCs by type:")
print(df["type"].value_counts().to_string())

COLORS = {"High": "#d62728", "Medium": "#ff9f1c", "Low": "#f2d34f", "Clean/Unknown": "#2ca02c", "No data": "#999999"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.2, 1]})
top = df.head(10).iloc[::-1]
ax1.barh([s[:38] for s in top["ioc"]], top["risk_score"], color=[COLORS[v] for v in top["verdict"]])
ax1.set_xlim(0, 100); ax1.set_xlabel("Combined risk score (0-100)")
ax1.set_title("Top IoCs by risk score" + ("  [DEMO DATA]" if DEMO_MODE else ""))

counts = df["verdict"].value_counts()
ax2.pie(counts, labels=counts.index, colors=[COLORS[v] for v in counts.index],
        autopct="%1.0f%%", startangle=90)
ax2.set_title("Verdict distribution")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "risk_report.png"), dpi=150)
plt.show()

## 10. Export results

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path  = os.path.join(OUTPUT_DIR, f"iocs_{stamp}.csv")
json_path = os.path.join(OUTPUT_DIR, f"iocs_{stamp}.json")
df.to_csv(csv_path, index=False)
df.to_json(json_path, orient="records", indent=2)
print("Saved:", csv_path, "|", json_path, "| chart: output/risk_report.png")

# High-risk IoCs ready to paste into a firewall / SIEM blocklist (defanged for safe sharing)
blocklist = df.loc[df["verdict"] == "High", "ioc"].tolist()
print("\nSuggested blocklist candidates (defanged):")
for b in blocklist:
    print("  ", b)

## 11. Limitations and next steps
* VirusTotal's free tier allows only 4 requests/minute, so large feeds take time (the rate limiter handles this).
* The scoring weights are simple heuristics – tune them for your environment and always review before blocking.
* Possible extensions: more feeds (AbuseIPDB, URLhaus), SQLite storage, scheduled runs, Slack/e-mail alerts, a Streamlit dashboard.